# Imputation of Nova Scores

In [2]:
import pickle
import json
import lightgbm as lgb
import numpy as np
from scipy.sparse import hstack, csr_matrix

# ── Load model ────────────────────────────────────────────────────────────────
gbm = lgb.Booster(model_file='outputs/nova_gbm_model.txt')

# ── Load encoders ─────────────────────────────────────────────────────────────
with open('outputs/nova_encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

grade_encoder = encoders['grade_encoder']
vectorizer    = encoders['vectorizer']

# ── Load feature metadata ─────────────────────────────────────────────────────
with open('outputs/nova_feature_metadata.json', 'r') as f:
    meta = json.load(f)

feature_names   = meta['feature_names']
additive_cols   = meta['additive_cols']
e_flag_cols     = meta['e_flag_cols']
frequent_e_nums = meta['frequent_e_nums']

# ── Build features for missing_nova and predict ───────────────────────────────
def build_sparse_feature_matrix(df_split, X_tfidf):
    grade     = csr_matrix(grade_encoder.transform(df_split[['nutriscore_grade']]))
    additives = csr_matrix(df_split[additive_cols].fillna(0).values)
    e_flags   = csr_matrix(df_split[e_flag_cols].fillna(0).values)
    return hstack([grade, additives, e_flags, X_tfidf])

X_tfidf_missing = vectorizer.transform(missing_nova['ingredients_clean'])
X_missing       = build_sparse_feature_matrix(missing_nova, X_tfidf_missing)

# predict and convert back to NOVA 1-4
preds_raw     = gbm.predict(X_missing)
preds_nova    = np.clip(np.round(preds_raw).astype(int), 0, 3) + 1

missing_nova  = missing_nova.copy()
missing_nova['nova_group_imputed'] = preds_nova

NameError: name 'missing_nova' is not defined